In [1]:
from helpers.data_preparation import * 
from helpers.nlp_functions import *
from helpers.time_series_functions import *

c:\Users\GSU\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [2]:
stock_list = stockListFromURL(market="usa")
stock_data = stockDataFromYf(stock_list)
price_data = stock_data["Close"][stock_list]

threshold = 0.3 * len(price_data)
price_data = price_data.dropna(thresh=len(price_data) - threshold, axis=1)
price_data = price_data.ffill()
price_data = price_data.bfill()

stock_list = list(price_data.columns)
stock2id = stocks2id(stock_list)
stock_data = stockDataFromYf(stock_list)

textual_data = fetchTextualInformation(stock_list, stock2id)

[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (period=5y)')
['BRK.B']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')
[*********************100%***********************]  496 of 496 completed
100%|██████████| 496/496 [02:52<00:00,  2.88it/s]


In [9]:
pd.DataFrame(textual_data).T.sample(5)

,ticker_id,code,name,sector,industry,description
HRL,230,HRL,Hormel Foods Corporation,Consumer Defensive,Packaged Foods,"Hormel Foods Corporation develops, processes, ..."
IRM,255,IRM,Iron Mountain Incorporated (Del,Real Estate,REIT - Specialty,Iron Mountain Incorporated (NYSE: IRM) is trus...
GOOGL,19,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...
JBL,257,JBL,Jabil Inc.,Technology,Electronic Components,Jabil Inc. provides manufacturing services and...
MSFT,311,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...


In [3]:
stock_list = textual_data.keys()

In [4]:
len(stock_list)

496

In [5]:
import json

best_n = {
    50: filterStocksByVolume(stock_data, largest_n=50).tolist(),
    100: filterStocksByVolume(stock_data, largest_n=100).tolist(),
    200: filterStocksByVolume(stock_data, largest_n=200).tolist(),
    300: filterStocksByVolume(stock_data, largest_n=300).tolist(),
    len(stock_list): list(stock_list)
}

# save best_n as json file
with open("best_n.json", "w") as f:
    json.dump(best_n, f, indent=4)

In [6]:
price_data.to_csv("price_data.csv")

In [ ]:
tokenizer, nlp_model = load_model()
description_feature_vectors = {}

for i, key in enumerate(list(textual_data.keys())):
    desc = textual_data[key]["description"]
    feature_vector = extract_features(text=desc, tokenizer=tokenizer, model=nlp_model)
    description_feature_vectors[i] = feature_vector

title = "usa"
with open(f'cold_data/description_feature_vectors{title}.pkl', 'wb') as f:
    pickle.dump(description_feature_vectors, f)

In [ ]:
start_date = "2020-08-21"
end_date = "2020-12-31"
window_sizes = [10, 20, 30, 40, 50]
overlaps = [1, 7, 14, 21]

for window_size in tqdm(window_sizes):
    for overlap in tqdm(overlaps):
        price_data = price_data[list(textual_data.keys())]
        sim_metrics = ["pearson", "dist_corr", "euclidean"] # TODO: add  dtw
        for metric in sim_metrics:
            ts_sim = calculate_historical_ts_similarities(price_data, start_date=start_date, end_date=end_date, 
                                                        similarity_metric=metric, 
                                                        window_size=window_size, overlap=overlap)

            with open(f'calculated_data/historical_ts_{metric}_{start_date}_{end_date}_{window_size}_{overlap}.pkl', 'wb') as f:
                pickle.dump(ts_sim, f)

  0%|          | 0/5 [00:00<?, ?it/s]

Date Range Count: 16









































  0%|          | 0/5 [00:18<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
price_data["Date"] = stock_data.index

price_data.to_csv(f"cold_data/stock_data{title}.csv", index=False)
pd.DataFrame(textual_data).to_csv(f"cold_data/textual_information{title}.csv", index=False)

In [80]:
s = pd.DataFrame(columns=list(price_data.drop("Date", axis=1).columns))
s.loc[0] = (list(stocks2id(stock_list).values()))
s.to_csv(f"cold_data/stock2id{title}.csv", index=False)